In [ ]:
import getpass
import html
import os
import re
import subprocess
from collections import defaultdict
from pathlib import Path

import soundfile as sf
import torch
from pyannote.audio import Pipeline
from yt_dlp import YoutubeDL
import unicodedata

URLS = [
    "https://www.youtube.com/watch?v=45jNv0XptsU",
]

OUTPUT_DIR = Path("data/YouTube_Videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: replace anonymous labels once you have identified the speakers.
# These mappings apply separately to each recording.
SPEAKER_NAMES = {
    # "SPEAKER_00": "Presenter",
    # "SPEAKER_01": "Guest",
}


################################################################################################
################################################################################################
################################################################################################
########### REPLACE WITH YOUR OWN TOKEN FROM https://huggingface.co/settings/tokens ############

HF_TOKEN = 'your_huggingface_token_here'

################################################################################################
################################################################################################
################################################################################################

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN,
)

if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))


# ---------- VTT parsing and rolling-caption deduplication ----------

TIMESTAMP = r"(?:\d{2}:)?\d{2}:\d{2}\.\d{3}"
CUE_TIMING = re.compile(rf"({TIMESTAMP})\s+-->\s+({TIMESTAMP})")
INLINE_TIMING = re.compile(rf"<({TIMESTAMP})>")


def seconds(timestamp):
    result = 0.0
    for part in timestamp.split(":"):
        result = result * 60 + float(part)
    return result


def clean_text(text):
    return html.unescape(re.sub(r"<[^>]*>", "", text))


def read_vtt_words(path):
    """Return (start, end, word), removing overlapping caption repetitions.

    Uses inline timestamps when present. Otherwise, word times are estimated
    by distributing words evenly within their caption interval.
    """
    content = path.read_text(encoding="utf-8-sig").replace("\r\n", "\n")
    words = []
    previous_end = -1.0

    for block in re.split(r"\n\s*\n", content):
        lines = block.splitlines()

        timing_index = next(
            (i for i, line in enumerate(lines) if CUE_TIMING.search(line)),
            None,
        )
        if timing_index is None:
            continue

        match = CUE_TIMING.search(lines[timing_index])
        cue_start, cue_end = map(seconds, match.groups())
        payload = " ".join(lines[timing_index + 1:])

        # Split caption text at YouTube's inline word timestamps.
        parts = INLINE_TIMING.split(payload)
        fragment_start = cue_start
        cue_words = []

        for i in range(0, len(parts), 2):
            fragment_end = (
                seconds(parts[i + 1]) if i + 1 < len(parts) else cue_end
            )
            tokens = clean_text(parts[i]).split()

            duration = max(0.0, fragment_end - fragment_start)
            for j, token in enumerate(tokens):
                start = fragment_start + duration * j / len(tokens)
                end = fragment_start + duration * (j + 1) / len(tokens)
                cue_words.append((start, end, token))

            fragment_start = fragment_end

        # YouTube repeats the preceding words in rolling captions.
        # Remove only a matching suffix/prefix near overlapping cue times.
        overlap = 0
        if words and cue_start <= previous_end + 0.05:
            max_overlap = min(len(words), len(cue_words))
            old_tokens = [word for _, _, word in words[-max_overlap:]]
            new_tokens = [word for _, _, word in cue_words]

            for k in range(max_overlap, 0, -1):
                if old_tokens[-k:] == new_tokens[:k]:
                    overlap = k
                    break

        words.extend(cue_words[overlap:])
        previous_end = cue_end

    return words


# ---------- Match transcript timestamps to diarization ----------

def speaker_for_interval(start, end, turns):
    """Choose the speaker with the greatest temporal overlap."""
    scores = defaultdict(float)

    for turn_start, turn_end, speaker in turns:
        if turn_start >= end:
            break
        if turn_end <= start:
            continue

        scores[speaker] += max(
            0.0, min(end, turn_end) - max(start, turn_start)
        )

    return max(scores, key=scores.get) if scores else "UNKNOWN"


def alceste_value(value):
    """Format metadata values using ASCII letters and digits."""
    value = unicodedata.normalize("NFKD", str(value))
    value = value.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^A-Za-z0-9]", "", value) or "unknown"


def write_transcript(words, turns, output_path, video_id):
    """Write an IRaMuTeQ corpus: one text per consecutive speaker turn."""
    blocks = []
    current_speaker = None
    current_words = []
    turn_number = 0

    def save_turn(speaker, tokens):
        nonlocal turn_number

        # Asterisks are reserved for corpus structure.
        # Preserve accents, apostrophes and punctuation in the speech.
        text = " ".join(tokens).replace("*", " ")
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            return

        turn_number += 1
        video = alceste_value(video_id)
        speaker_code = alceste_value(speaker)

        # Scope speaker identity to its recording:
        # SPEAKER_00 in two videos need not be the same person.
        participant = f"v{video}s{speaker_code}"

        header = (
            f"**** *video_{video}"
            f" *locuteur_{participant}"
            f" *tour_{turn_number:05d}"
        )

        if speaker in SPEAKER_NAMES:
            name = alceste_value(SPEAKER_NAMES[speaker])
            header += f" *nom_{name}"

        blocks.append(f"{header}\n{text}")

    for start, end, word in words:
        speaker = speaker_for_interval(start, end, turns)

        if speaker != current_speaker:
            if current_words:
                save_turn(current_speaker, current_words)

            current_speaker = speaker
            current_words = []

        current_words.append(word)

    if current_words:
        save_turn(current_speaker, current_words)

    output_path.write_text(
        "\n\n".join(blocks) + ("\n" if blocks else ""),
        encoding="utf-8",
    )


# ---------- Download and process ----------

download_options = {
    "format": "bestaudio",
    "noplaylist": True,
    "writesubtitles": True,
    "writeautomaticsub": True,
    "subtitleslangs": ["fr-orig"],
    "subtitlesformat": "vtt",
    "sleep_interval_subtitles": 10,
    "outtmpl": str(OUTPUT_DIR / "%(id)s.%(ext)s"),
}

for url in URLS:
    if not url.strip():
        continue

    with YoutubeDL(download_options) as ydl:
        info = ydl.extract_info(url, download=True)
        source_audio = Path(ydl.prepare_filename(info))

    video_id = info["id"]
    vtt_path = OUTPUT_DIR / f"{video_id}.fr-orig.vtt"
    wav_path = OUTPUT_DIR / f"{video_id}.16k.wav"
    txt_path = OUTPUT_DIR / f"{video_id}.txt"

    if not vtt_path.exists():
        raise FileNotFoundError(
            f"No fr-orig VTT found for {video_id}. "
            "Check available languages with yt-dlp --list-subs URL."
        )

    # Keep the original timeline: no trimming or silence removal.
    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(source_audio),
            "-vn", "-ac", "1", "-ar", "16000",
            "-c:a", "pcm_s16le",
            str(wav_path),
        ],
        check=True,
    )

    # Load the WAV directly to avoid relying on pyannote's audio decoder.
    audio, sample_rate = sf.read(
        wav_path, dtype="float32", always_2d=True
    )
    waveform = torch.from_numpy(audio.T.copy())

    print(f"Diarizing {video_id}...")
    result = pipeline({
        "waveform": waveform,
        "sample_rate": sample_rate,
    })

    turns = sorted(
        (turn.start, turn.end, speaker)
        for turn, _, speaker
        in result.exclusive_speaker_diarization.itertracks(yield_label=True)
    )

    words = read_vtt_words(vtt_path)
    if not words:
        raise ValueError(f"No transcript text found in {vtt_path}")

    write_transcript(words, turns, txt_path, video_id)
    print(f"Saved: {txt_path}")